In [2]:
import sys
import os

# Thêm thư mục cha (thư mục gốc chứa folder algorithms) vào hệ thống
sys.path.append(os.path.abspath(os.path.join('..')))
import pyspark.sql.functions as F
from algorithms.fpgrowthsuper import FPGrowthSparkMultiSupport
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

def run_comparison_test(spark):
    input_file = '../asso_products.parquet'
        # ====================================================================
    # PHẦN 2: CHẠY MÔ HÌNH ĐA NGƯỠNG (MULTI-SUPPORT)
    # ====================================================================
    print("\n\n" + "="*85)
    print(" 2. CHẠY MÔ HÌNH FP-GROWTH ĐA NGƯỠNG (RÀNG BUỘC CỰC ĐẠI) ".center(85))
    print("="*85)

    # Đọc dữ liệu bằng Spark
    print(f"--- Đang phân tích dữ liệu từ '{input_file}'... ---")
    df = spark.read.parquet(input_file)
    
    total_transactions = df.count()
    print(f" ĐANG PHÂN TÍCH TẦN SUẤT SẢN PHẨM TRÊN {total_transactions} GIAO DỊCH ".center(85))
    print("-" * 85)

    # 1. Đếm tần suất thực tế của tất cả sản phẩm bằng PySpark (thay vì Pandas)
    item_counts_df = df.select(F.explode("items").alias("item")).groupBy("item").count()
    
    # Đưa về Pandas để xử lý dictionary cho nhanh (vì dict này rất nhỏ, không tốn RAM)
    item_counts_pd = item_counts_df.toPandas()
    item_counts_pd['support'] = item_counts_pd['count'] / total_transactions
    
    actual_supports = dict(zip(item_counts_pd['item'], item_counts_pd['support']))

    print(f"Tổng số loại sản phẩm phân biệt: {len(actual_supports)}")

    # 2. Thuật toán tự động phân cụm và gán min_support theo logic của ông
    item_supports_dict = {}
    count_high = count_med = count_low = 0

    for item, supp in actual_supports.items():
        if supp >= 0.02:        
            item_supports_dict[item] = 0.020
            count_high += 1
        elif supp >= 0.017:     
            item_supports_dict[item] = 0.015
            count_med += 1
        else:                   
            item_supports_dict[item] = 0.01
            count_low += 1

    print("\nĐÃ TỰ ĐỘNG GÁN NGƯỠNG RÀNG BUỘC CỰC ĐẠI:")
    print(f"- Nhóm Phổ biến (min_support = 0.02): {count_high} sản phẩm")
    print(f"- Nhóm Trung bình (min_support = 0.015): {count_med} sản phẩm")
    print(f"- Nhóm Hàng hiếm  (min_support = 0.010): {count_low} sản phẩm")
    print("-" * 65)


    fpgrowth_super_model = FPGrowthSparkMultiSupport(spark, input_path=input_file)
    # Truyền dictionary ngưỡng vào hàm build
    fpgrowth_super_model.build_fp_tree_and_mine(
        item_supports=item_supports_dict, 
        min_confidence=0.5,
        metric='lift',
        min_threshold=2.0
    )
    # Lưu kết quả
    fpgrowth_super_model.save_rules(output_path="../data/results/association_rules_super.parquet")
    # 4. Hiển thị kết quả của mô hình siêu việt
    fpgrowth_super_model.display_top_itemsets(top_n=100)
    fpgrowth_super_model.display_top_rules(top_n=100)
    
    if fpgrowth_super_model.rules is not None:
        print(f" > Tổng số luật sinh ra sau khi lọc khắt khe: {fpgrowth_super_model.rules.count()}")

# NẾU CHẠY TRONG JUPYTER NOTEBOOK:
run_comparison_test(spark)



               2. CHẠY MÔ HÌNH FP-GROWTH ĐA NGƯỠNG (RÀNG BUỘC CỰC ĐẠI)               
--- Đang phân tích dữ liệu từ '../asso_products.parquet'... ---
                ĐANG PHÂN TÍCH TẦN SUẤT SẢN PHẨM TRÊN 23204 GIAO DỊCH                
-------------------------------------------------------------------------------------
Tổng số loại sản phẩm phân biệt: 3768

ĐÃ TỰ ĐỘNG GÁN NGƯỠNG RÀNG BUỘC CỰC ĐẠI:
- Nhóm Phổ biến (min_support = 0.02): 223 sản phẩm
- Nhóm Trung bình (min_support = 0.015): 89 sản phẩm
- Nhóm Hàng hiếm  (min_support = 0.010): 3456 sản phẩm
-----------------------------------------------------------------
--- Đang đọc dữ liệu từ ../asso_products.parquet... ---
--- Đang khai phá thô với minSupport thấp nhất: 0.01 ---
--- Đang áp dụng bộ lọc Đa ngưỡng (Multi-Support) ---
--- Đang lưu luật kết hợp... ---


 [+] Đã lưu luật tại: ../data/results/association_rules_super.parquet (Parquet/CSV/JSON)


 TOP 100 TẬP MỤC PHỔ BIẾN =============================
                            itemsets                              freq  support
                              Cream Hanging Heart T-Light Holder  2311  0.0996 
                                        Regency Cakestand 3 Tier  2169  0.0935 
                                         Jumbo Bag Red Retrospot  2135  0.0920 
                                                   Party Bunting  1706  0.0735 
                                         Lunch Bag Red Retrospot  1608  0.0693 
                                   Assorted Colour Bird Ornament  1467  0.0632 
                                Set Of 3 Cake Tins Pantry Design  1458  0.0628 
                                                  Popcorn Holder  1442  0.0621 
                                 Pack Of 72 Retrospot Cake Cases  1334  0.0575 
                                           Lunch Bag Suki Design  1306  0.0563 
                                           Lunch Bag Black Skull